# Krauss–Do–Huck (2017) — simplified classroom pipeline

**Paper:** Krauss, C., Do, X. A., & Huck, N. (2017). *Deep neural networks, gradient-boosted trees, random forests: Statistical arbitrage on the S&P 500.* *European Journal of Operational Research*, 259(2), 689–702.

Two data versions, **same pipeline** (label → models → long–short → costs):

| `DATA_MODE` | What you get |
|-------------|--------------|
| `"synthetic"` | Planted nonlinear signal (readable AUC/Sharpe in minutes) |
| `"real"` | Yahoo CSV: SPY + large caps (`data/krauss_sp500_daily.csv`) |

**Label (paper §4.2):** \(y=1\) if the next-day return is **above the cross-sectional median** of all stocks that day — not vs SPY. The abstract’s “outperform the general market” is loose wording.

Set the switch in the next code cell. (We discuss this result in class.)

**Companion:** `sessions/02_krauss_do_huck.md`


## 0. Setup + choose data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")

# --- switch + B2 levers ---
DATA_MODE = "real"   # "synthetic" or "real"
K_PER_SIDE = None    # default max(1, round(0.02 * n_names)); set an int to override
COST_BPS = 5         # half-turn cost in basis points (paper: 5)
LABEL = "median"     # "median" = beat the cross-section; "index" = beat SPY
MODELS = ["Logit", "RF", "GBT", "MLP"]  # subset to run fewer models
# ----------------

rng = np.random.default_rng(2017)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
assert DATA_MODE in {"synthetic", "real"}, DATA_MODE
assert LABEL in {"median", "index"}, LABEL
assert MODELS, MODELS


def display_path(p):
    p = Path(p).resolve()
    for base in p.parents:
        if (base / "requirements.txt").is_file():
            try:
                return p.relative_to(base).as_posix()
            except ValueError:
                break
    return p.name

print("DATA_MODE =", DATA_MODE, "| LABEL =", LABEL, "| COST_BPS =", COST_BPS, "| MODELS =", MODELS)

## 1. Panel

**Features (paper §3):** 31 lagged simple returns \(R_{t,m}\) with \(m \in \{1,\dots,20\} \cup \{40, 60, \dots, 240\}\).

**Label (paper §4.2):** \(y=1\) if next-day return is above that day’s **cross-sectional median** — not vs SPY.

**k:** `k = max(1, round(0.02 * n_names))` so both modes trade ~2% per side (paper: 10 of ~500). Synthetic: 80 names → `k = 2`. Real: 50 names → `k = 1`. (`k = 10` of 80 would be 12.5% per side, not 2%.)


In [ ]:
# Paper §3: 31 lagged simple returns R_{t,m}, m in {1..20} U {40,60,...,240}.
LAGS = list(range(1, 21)) + list(range(40, 241, 20))
assert len(LAGS) == 31, len(LAGS)
features = [f"R_{m}" for m in LAGS]


def add_return_lags(df, id_col, close_col="close"):
    out = df.copy()
    g = out.groupby(id_col, group_keys=False)[close_col]
    for m in LAGS:
        out[f"R_{m}"] = g.transform(lambda s, m=m: s / s.shift(m) - 1.0)
    return out


if DATA_MODE == "synthetic":
    N, T = 80, 900
    dates = pd.bdate_range("2015-01-01", periods=T)
    stocks = np.arange(N)
    r_m = rng.normal(loc=0.0002, scale=0.01, size=T)
    lag1 = np.zeros(N)
    lag5 = np.zeros(N)
    mom20 = np.zeros(N)
    rows = []
    for t, dt in enumerate(dates):
        z1, z5, z20 = lag1.copy(), lag5.copy(), mom20.copy()
        z_cross = z1 * z5
        score = (
            0.8 * (z1 / 0.02)
            + 0.6 * np.tanh(z20 / 0.02)
            + 1.0 * np.tanh(5.0 * z_cross / (0.02 ** 2))
            + 0.5 * ((z5 / 0.02) ** 2 - 1.0)
        )
        alpha = 0.012 * np.tanh(score)
        r_i = r_m[t] + alpha + rng.normal(scale=0.012, size=N)
        rows.append(pd.DataFrame({
            "date": dt, "stock": stocks, "ret": r_i,
            "excess": r_i - r_m[t], "alpha_true": alpha,
        }))
        lag1 = r_i - r_m[t]
        lag5 = 0.8 * lag5 + 0.2 * lag1
        mom20 = 0.9 * mom20 + 0.1 * lag1
    panel = pd.concat(rows, ignore_index=True)
    panel["close"] = panel.groupby("stock")["ret"].transform(lambda s: (1.0 + s).cumprod())
    panel = add_return_lags(panel, "stock")
    g = panel.groupby("stock", group_keys=False)
    panel["ret_fwd"] = g["ret"].shift(-1)
    panel["alpha_true"] = g["alpha_true"].shift(-1)
    if LABEL == "median":
        panel["y_beat"] = panel.groupby("date")["ret_fwd"].transform(
            lambda s: (s > s.median()).astype(int) if s.notna().any() else np.nan
        )
    else:
        panel["mkt"] = panel["ret"] - panel["excess"]
        panel["mkt_fwd"] = panel.groupby("stock")["mkt"].shift(-1)
        panel["y_beat"] = np.where(
            panel["ret_fwd"].notna() & panel["mkt_fwd"].notna(),
            (panel["ret_fwd"] > panel["mkt_fwd"]).astype(int),
            np.nan,
        )
    panel["excess_fwd"] = panel["ret_fwd"]
    panel = panel.dropna(subset=features + ["y_beat", "excess_fwd"]).copy()
    train_end = pd.Timestamp("2016-12-30")
    val_end = pd.Timestamp("2017-06-30")
else:
    candidates = [Path("data/krauss_sp500_daily.csv"), Path("../data/krauss_sp500_daily.csv")]
    DATA = next((p for p in candidates if p.exists()), None)
    if DATA is None:
        raise FileNotFoundError(
            "Price file not found. Run: python data/download_krauss_panel.py, then rerun this notebook. "
            'Or set DATA_MODE = "synthetic" in the setup cell and rerun.'
        )
    print("using", display_path(DATA))
    px = pd.read_csv(DATA, parse_dates=["date"]).sort_values(["ticker", "date"])
    panel = px.loc[px["ticker"] != "SPY"].copy()
    panel["stock"] = panel["ticker"]
    panel["ret"] = panel.groupby("stock")["close"].pct_change()
    panel = add_return_lags(panel, "stock")
    g = panel.groupby("stock", group_keys=False)
    panel["ret_fwd"] = g["ret"].shift(-1)
    if LABEL == "median":
        panel["y_beat"] = (
            panel.groupby("date")["ret_fwd"].transform(
                lambda s: (s > s.median()).astype(int) if s.notna().any() else np.nan
            )
        )
    else:
        spy = px.loc[px["ticker"] == "SPY", ["date", "close"]].sort_values("date")
        spy["spy_fwd"] = spy["close"].pct_change().shift(-1)
        panel = panel.merge(spy[["date", "spy_fwd"]], on="date", how="left")
        panel["y_beat"] = np.where(
            panel["ret_fwd"].notna() & panel["spy_fwd"].notna(),
            (panel["ret_fwd"] > panel["spy_fwd"]).astype(int),
            np.nan,
        )
    panel["excess_fwd"] = panel["ret_fwd"]
    panel = panel.dropna(subset=features + ["y_beat", "excess_fwd"]).copy()
    train_end = pd.Timestamp("2018-12-31")
    val_end = pd.Timestamp("2020-06-30")

n_univ = int(panel["stock"].nunique())
k = max(1, int(round(0.02 * n_univ))) if K_PER_SIDE is None else int(K_PER_SIDE)
print("features: 31 lagged simple returns R_m, m =", LAGS)
print(panel.shape, "| P(y=1) =", round(panel["y_beat"].mean(), 3))
print("dates:", panel["date"].min().date(), "→", panel["date"].max().date())
print(f"k = {k}  ({k / n_univ:.1%} of a {n_univ}-name universe)")
if DATA_MODE == "real":
    print("Paper: k = 10 of ~500 S&P names (~2%). This panel is current large-caps, not the historical index.")
    print("n/a: the real sample starts in 2015")
panel.head()


**Real-mode caveat.** The 50 Yahoo names are **today’s survivors** (current large-caps, not the historical S&P 500). The paper uses monthly constituent lists (Section 3.1) and trades \(k=10\) of ~500 names (~2%). Here default \(k=1\) is 2% of a winner-biased universe — closer on the share, not a replication.


## 2. Chronological split

In [ ]:
train = panel[panel["date"] <= train_end]
val = panel[(panel["date"] > train_end) & (panel["date"] <= val_end)]
test = panel[panel["date"] > val_end]

for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s}: {df['date'].nunique():4d} days | {len(df):7d} rows | pos={df['y_beat'].mean():.3f}")

X_train, y_train = train[features].to_numpy(), train["y_beat"].to_numpy()
X_val, y_val = val[features].to_numpy(), val["y_beat"].to_numpy()
X_test, y_test = test[features].to_numpy(), test["y_beat"].to_numpy()

## 3. Model ladder

In [ ]:
candidates = {
    "Logit": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=500, C=1.0)),
    ]),
    "RF": RandomForestClassifier(
        n_estimators=200, max_depth=5, min_samples_leaf=40,
        random_state=2017, n_jobs=-1,
    ),
    "GBT": GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05, random_state=2017,
    ),
    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(32, 16), activation="relu",
            alpha=1e-3, max_iter=400, random_state=2017,
        )),
    ]),
}
missing = [m for m in MODELS if m not in candidates]
assert not missing, missing
candidates = {m: candidates[m] for m in MODELS}

fitted = {}
val_rows = []
for name, model in candidates.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    proba = model.predict_proba(X_val)[:, 1]
    pred = (proba >= 0.5).astype(int)
    val_rows.append({
        "model": name,
        "acc_val": accuracy_score(y_val, pred),
        "auc_val": roc_auc_score(y_val, proba),
    })
    print(f"{name:5s}  val AUC={val_rows[-1]['auc_val']:.3f}  acc={val_rows[-1]['acc_val']:.3f}")

val_leaderboard = pd.DataFrame(val_rows).sort_values("auc_val", ascending=False)
val_leaderboard.round(3)

## 4. Long–short from probabilities

P&L column is `excess_fwd` (next-day **simple return** in **real** mode; same-day excess in **synthetic**). Equal-weight long–short is the market-neutral book that matches the median label.

Daily cost = \(4 \times\) turnover \(\times\) 5 bp: two half-turns per replaced name \(\times\) two legs. At full turnover this is the paper’s **0.20% per day** (0.45% → 0.25%).

In [ ]:
def daily_long_short(df, proba, k=1, cost_bps=0.0):
    """Equal-weight long top-k / short bottom-k.
    Turnover = average share of names replaced per leg.
    Daily cost = 4 × turnover × cost_bps (2 half-turns × 2 legs).
    """
    out = df[["date", "stock", "excess_fwd"]].copy()
    out["proba"] = proba
    port = []
    prev_long, prev_short = set(), set()
    for dt, g in out.groupby("date"):
        g = g.sort_values("proba")
        if len(g) < 2 * k:
            continue
        short = set(g.head(k)["stock"])
        long = set(g.tail(k)["stock"])
        r_long = g.loc[g["stock"].isin(long), "excess_fwd"].mean()
        r_short = g.loc[g["stock"].isin(short), "excess_fwd"].mean()
        turn = 0.0
        if prev_long or prev_short:
            turn = 0.5 * (len(long - prev_long) / k + len(short - prev_short) / k)
        cost = 4.0 * turn * (cost_bps / 1e4)
        gross = r_long - r_short
        port.append({"date": dt, "ls_gross": gross, "ls": gross - cost, "turn": turn})
        prev_long, prev_short = long, short
    return pd.DataFrame(port).set_index("date").sort_index()


def sharpe_daily(x):
    mu, sd = x.mean(), x.std(ddof=1)
    return np.sqrt(252) * mu / sd if sd > 0 else np.nan


def break_even_bps(ls):
    turn = ls["turn"].mean()
    if turn <= 0:
        return np.nan
    return ls["ls_gross"].mean() / (4.0 * turn) * 1e4


rows = []
ls_paths = {}
print(f"k = {k} of {test['stock'].nunique()} names "
      f"({k / test['stock'].nunique():.1%} of this universe; paper ≈ 10/500 = 2%)")
for name, model in fitted.items():
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    ls0 = daily_long_short(test, proba, k=k, cost_bps=0.0)
    ls5 = daily_long_short(test, proba, k=k, cost_bps=COST_BPS)
    ls_paths[name] = ls5["ls"]
    rows.append({
        "model": name,
        "acc_test": accuracy_score(y_test, pred),
        "auc_test": roc_auc_score(y_test, proba),
        "ls_mean_before": ls0["ls"].mean(),
        "ls_mean_5bp": ls5["ls"].mean(),
        "break_even_bps": break_even_bps(ls0),
        "ls_sharpe_ann_5bp": sharpe_daily(ls5["ls"]),
        "avg_turnover": ls0["turn"].mean(),
    })

if DATA_MODE == "synthetic" and "alpha_true" in test.columns:
    ls0 = daily_long_short(test, test["alpha_true"].to_numpy(), k=k, cost_bps=0.0)
    ls5 = daily_long_short(test, test["alpha_true"].to_numpy(), k=k, cost_bps=COST_BPS)
    ls_paths["ORACLE"] = ls5["ls"]
    rows.append({
        "model": "ORACLE",
        "acc_test": np.nan,
        "auc_test": roc_auc_score(y_test, test["alpha_true"]),
        "ls_mean_before": ls0["ls"].mean(),
        "ls_mean_5bp": ls5["ls"].mean(),
        "break_even_bps": break_even_bps(ls0),
        "ls_sharpe_ann_5bp": sharpe_daily(ls5["ls"]),
        "avg_turnover": ls0["turn"].mean(),
    })

results = pd.DataFrame(rows).sort_values("ls_sharpe_ann_5bp", ascending=False)
print("Costs: 5 bp per half-turn; daily cost = 4 × turnover × 5 bp. "
      "At full turnover that is 0.20%/day (paper: 0.45% → 0.25%).")
results.round(4)


In [ ]:
cum = pd.DataFrame({m: (1 + s).cumprod() for m, s in ls_paths.items()})
ax = cum.plot(title=f"Cumulative LS (test), mode={DATA_MODE}, k={k}, 5bp half-turn")
ax.set_ylabel("Growth of $1")
plt.tight_layout()
plt.show()


## 5. Costs

In [ ]:
best = val_leaderboard.iloc[0]["model"]
proba_best = fitted[best].predict_proba(X_test)[:, 1]
ls0 = daily_long_short(test, proba_best, k=k, cost_bps=0.0)
print(f"best-on-val = {best}")
print("mean daily LS before costs:", round(ls0["ls"].mean(), 5))
print("mean daily LS after 5 bp:", round(daily_long_short(test, proba_best, k=k, cost_bps=5)["ls"].mean(), 5))
print("break-even half-turn cost (bp):", round(break_even_bps(ls0), 2))
cost_rows = []
for c in [0, 1, 5, 10, 20]:
    ls = daily_long_short(test, proba_best, k=k, cost_bps=c)
    cost_rows.append({
        "cost_bps": c,
        "ls_mean_daily": ls["ls"].mean(),
        "ls_sharpe_ann": sharpe_daily(ls["ls"]),
        "avg_turnover": ls["turn"].mean(),
    })
pd.DataFrame(cost_rows).round(4)


## 6. Discussion

1. Flip `DATA_MODE` and re-run: what happens to AUC?
2. Why can a planted DGP mislead a PhD student about live trading?
3. After costs, do you still trade the real-data book?

## 7. Experiments

In [ ]:
k_alt = 2 if DATA_MODE == "real" else 20
proba = fitted[best].predict_proba(X_test)[:, 1]
ls_alt = daily_long_short(test, proba, k=k_alt, cost_bps=5)
print(f"mode={DATA_MODE} model={best}, k={k_alt} ({k_alt / test['stock'].nunique():.1%}), cost=5bps")
print("mean daily LS:", round(ls_alt["ls"].mean(), 5))
print("ann. Sharpe:  ", round(sharpe_daily(ls_alt["ls"]), 3))


## Takeaway

(We discuss this result in class.)
